# [SK 09 - Multi-Agent orchestration](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-orchestration/magentic?source=recommendations&pivots=programming-language-python)
Here we exploit [YAML declarative specification](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python#declarative-spec), configurable with:
- chat_completion_agent
- foundry_agent
- azure_assistant
- azure_responses
- openai_assistant
- openai_responses

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

project_endpoint = os.environ["AIF_STD_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.1.0b4
azure-ai-agents library installed version: 1.2.0b6


# Helper functions

## Universal `ChatWithAgentStreamAsync` - for single agents
The following function works with any SK agent (built from ChatCompletion / Assistant / Response / AI Foundry) implementing a streaming response.<br/>
It is **not** used for conversations in group chats.

In [2]:
async def ChatWithAgentStreamAsync(agent, USER_INPUTS: list) -> None:
    # return
    from semantic_kernel.agents import AzureAIAgentThread
    from semantic_kernel.contents import AuthorRole
    
    i=0
    for user_input in USER_INPUTS:
        print(f"\n************************************\nMessage {i} from {AuthorRole.USER}: '{user_input}'")
        # Invoke the agent for the specified task
        is_code = False
        last_role = None
        async for response in agent.invoke_stream(
            messages=user_input,
        ):
            current_is_code = response.metadata.get("code", False)

            if current_is_code:
                if not is_code:
                    print("\n\n```python")
                    is_code = True
                print(response.content, end="", flush=True)
            else:
                if is_code:
                    print("\n```")
                    is_code = False
                    last_role = None
                if hasattr(response, "role") and response.role is not None and last_role != response.role:
                    print(f"\n# {response.role}: ", end="", flush=True)
                    last_role = response.role
                print(response.content, end="", flush=True)
        if is_code:
            print("```\n")
        print()

## Callback function to observe agent responses in group chats

In [3]:
from semantic_kernel.contents import ChatMessageContent

# Track the last agent name to avoid repeating it unnecessarily
last_agent_name = None
first_print = True

def agent_response_callback(message: ChatMessageContent) -> None:
    global last_agent_name
    global first_print

    # Print agent name only when it changes
    if message.name != last_agent_name:
        if first_print:
            first_print = False
        else:
            print(f"\n\n\n")

        if not message.name is None: # sometimes we get this nonsense agent, which we don't print
            print(f"==> AGENT **{message.name}**---\n", end="", flush=True)
            
        last_agent_name = message.name

    # Stream content inline
    print(message.content, end="", flush=True)

# Agent definitions 

## *Research* Agent - AI Foundry with Bing Search

### Create AI Foundry Project Client (AIProjectClient)

In [4]:
from semantic_kernel.agents import AzureAIAgent
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())

### Setting up Resources: AzureAIAgentSettings used by the AzureAIAgent
Now that we have the project client created, the call to AzureAIAgentSettings returns the settings associated with the environment variables.
If we do it before creating the project client, it does not capture all the proper settings.

In [5]:
from semantic_kernel.agents import AzureAIAgentSettings

aiagent_settings = AzureAIAgentSettings()
aiagent_settings

AzureAIAgentSettings(env_file_path=None, env_file_encoding='utf-8', model_deployment_name='gpt-4o', endpoint='https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2', agent_id=None, bing_connection_id=None, azure_ai_search_connection_id=None, azure_ai_search_index_name=None, api_version=None, deep_research_model=None)

### Retrieve the connection id for the Bing Grounding resource

In [6]:
bingconnection_id = ""

async for c in project_client.connections.list():
    if c.name == os.environ["BING_GROUNDING_CONNECTION_NAME"]:
        bingconnection_id = c.id

print(f"Bing connection id: {bingconnection_id}\n")

Bing connection id: /subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif2stdrg/providers/Microsoft.CognitiveServices/accounts/aif2stdsvhdu2/projects/aif2stdwusprj01hdu2/connections/groundingwithbingsearch



### Load the agent definition

In [7]:
# Read the agent template from the file
with open("./_agents/magentic_01_researcher.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
research_agent_specs = eval(f"f'''{fstring_template}'''")
print(research_agent_specs)

type: foundry_agent
name: ResearchAgent
description: A helpful assistant with access to web search. Ask it to perform web searches.
model:
  id: gpt-4o
  options:
    temperature: 0.4
tools:
  - type: bing_grounding
    options:
      tool_connections:
        - /subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif2stdrg/providers/Microsoft.CognitiveServices/accounts/aif2stdsvhdu2/projects/aif2stdwusprj01hdu2/connections/groundingwithbingsearch
instructions: >
  You are a Researcher. You find information without additional computation or quantitative analysis.


### Create the Semantic Kernel Agent, based on Azure AI Foundry Agent

In [8]:
from semantic_kernel.agents import AgentRegistry

research_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=research_agent_specs,
    client=project_client,
    settings=aiagent_settings,
)
research_agent

AzureAIAgent(arguments={'temperature': 0.4}, description='A helpful assistant with access to web search. Ask it to perform web searches.', id='asst_TjeIuVF5Ja3d7aZvRiFYILYF', instructions='You are a Researcher. You find information without additional computation or quantitative analysis.', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000013D9E36C1A0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='ResearchAgent', prompt_template=None, client=<azure.ai.projects.aio._patch.AIProjectClient object at 0x0000013D9BAA1A90>, definition={'id': 'asst_TjeIuVF5Ja3d7aZvRiFYILYF', 'object': 'assistant', 'created_at': 1761577674, 'name': 'ResearchAgent', 'description': 'A helpful assistant with access to web search. Ask it to perform web searches.', 'model': 'gpt-4o', 'instructions': 'You are a Researcher. You f

### Invoke the agent

In [9]:
RESEARCH_USER_INPUTS = [
    "What is the smallest reptile?", 
    "Generate a mathematical question like 'How much is pi ^ (-3.1)'. Do not use 'e'",
]

await ChatWithAgentStreamAsync(research_agent, RESEARCH_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'What is the smallest reptile?'

# AuthorRole.ASSISTANT: The smallest reptile known is the Brookesia nana, also referred to as the nano-chameleon. It was discovered in northern Madagascar and first described in 2021. An adult male Brookesia nana measures only about 13.5 mm (0.53 inches) from the snout to the base of the tail, making it the smallest known adult reptile in the world.

************************************
Message 0 from AuthorRole.USER: 'Generate a mathematical question like 'How much is pi ^ (-3.1)'. Do not use 'e''

# AuthorRole.ASSISTANT: Sure, here's a mathematical question for you:

What is the value of \( 5^{(-2.5)} \)?


## *Code* Agent - OpenAI Assistant with Code Interpreter

### Load the Agent definition

In [10]:
# Read the agent template from the file
with open("./_agents/magentic_02_coder.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
coder_agent_specs = eval(f"f'''{fstring_template}'''")
print(coder_agent_specs)

type: azure_assistant
name: CoderAgent
description: A helpful assistant that writes and executes code to process and analyze data.
model:
  id: gpt-4o
  options:
    temperature: 0.4
tools:
  - type: code_interpreter
instructions: >
  You solve questions using code. Please provide detailed analysis and computation process.


### Create the assistant client

In [11]:
from semantic_kernel.agents import AzureAssistantAgent
assistant_client = AzureAssistantAgent.create_client()
print(f"Assistant base URL: {assistant_client.base_url}")

Assistant base URL: https://mmoaiswc-01.openai.azure.com/openai/


### Create the assistant agent

In [12]:
from semantic_kernel.agents import AgentRegistry

coder_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=coder_agent_specs,
    client=assistant_client
)
coder_agent

AzureAssistantAgent(arguments={'temperature': 0.4}, description='A helpful assistant that writes and executes code to process and analyze data.', id='asst_wE84sKJvYgOldB2CwS0WwVl5', instructions='You solve questions using code. Please provide detailed analysis and computation process.', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000013D9BB934D0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='CoderAgent', prompt_template=None, client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000013D9E36DD30>, definition=Assistant(id='asst_wE84sKJvYgOldB2CwS0WwVl5', created_at=1761577686, description='A helpful assistant that writes and executes code to process and analyze data.', instructions='You solve questions using code. Please provide detailed analysis and computation process.', metadata={}, model='gp

### Invoke the agent

In [13]:
STATISTICIAN_USER_INPUTS = [
    "Calculate how much is pi ^ -3.1", #  right answer is 0.02876
]

await ChatWithAgentStreamAsync(coder_agent, STATISTICIAN_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'Calculate how much is pi ^ -3.1'


```python
import math

# Calculate pi raised to the power of -3.1
result = math.pi ** -3.1
result
```

# AuthorRole.ASSISTANT: The value of \(\pi^{-3.1}\) is approximately \(0.02876\).


# Agents collection into a list

In [14]:
def get_agents():
    agents = [research_agent, coder_agent]
    return agents

agents = get_agents()
agents

[AzureAIAgent(arguments={'temperature': 0.4}, description='A helpful assistant with access to web search. Ask it to perform web searches.', id='asst_TjeIuVF5Ja3d7aZvRiFYILYF', instructions='You are a Researcher. You find information without additional computation or quantitative analysis.', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000013D9E36C1A0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='ResearchAgent', prompt_template=KernelPromptTemplate(prompt_template_config=PromptTemplateConfig(name='', description='', template='You are a Researcher. You find information without additional computation or quantitative analysis.', template_format='semantic-kernel', input_variables=[], allow_dangerously_set_content=False, execution_settings={}), allow_dangerously_set_content=False), client=<azure.ai.p

# Groups Chat Types

## [Sequential Chat](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-orchestration/sequential?pivots=programming-language-python)

### Set Up the Sequential Orchestration
SequentialOrchestration object, passing in the agents and the optional response callback.

In [15]:
from semantic_kernel.agents import SequentialOrchestration

agents = get_agents()
sequential_orchestration = SequentialOrchestration(
    members=agents,
    agent_response_callback=agent_response_callback,
)

### Start the Runtime
Start the runtime to manage agent execution

In [16]:
from semantic_kernel.agents.runtime import InProcessRuntime

sequential_runtime = InProcessRuntime()
sequential_runtime.start()

==> AGENT **ResearchAgent**---
Sure, here is a mathematical question similar in style but not using 'e':

What is the value of \( \sqrt{2}^{-4.5} \)?



==> AGENT **CoderAgent**---
# Calculating the value of sqrt(2) raised to the power of -4.5

import math

# Calculating the result
value = math.sqrt(2) ** -4.5
valueThe value of \( \sqrt{2}^{-4.5} \) is approximately \( 0.2102 \). 

To elaborate on the computation:

1. **Calculating \(\sqrt{2}\):** This is the square root of 2, which is approximately \( 1.4142 \).

2. **Raising to a Negative Power:** Raising a number to a negative exponent corresponds to taking the reciprocal of the number raised to the positive of that exponent. So, \( \sqrt{2}^{-4.5} = \frac{1}{\sqrt{2}^{4.5}} \).

3. **Computing \(\sqrt{2}^{4.5}\):** This exponentiates the square root of 2 to the power 4.5, resulting in approximately \( 4.7568 \).

4. **Taking the Reciprocal:** Finally, \( \frac{1}{4.7568} \) gives approximately \( 0.2102 \).

### Invoke the Orchestration
Invoke the orchestration with your initial task (e.g., a product description). The output will flow through each agent in sequence.

In [17]:
sequential_orchestration_result = await sequential_orchestration.invoke(
    task="Generate a mathematical question like 'How much is pi ^ (-3.1)'. Do not use 'e'",
    runtime=sequential_runtime,
)

### Collect Results
Wait for the orchestration to complete and print the final result.

In [18]:
value = await sequential_orchestration_result.get()
print(f"\nFinal result:\n{value}")


Final result:
# Calculating the value of sqrt(2) raised to the power of -4.5

import math

# Calculating the result
value = math.sqrt(2) ** -4.5
valueThe value of \( \sqrt{2}^{-4.5} \) is approximately \( 0.2102 \). 

To elaborate on the computation:

1. **Calculating \(\sqrt{2}\):** This is the square root of 2, which is approximately \( 1.4142 \).

2. **Raising to a Negative Power:** Raising a number to a negative exponent corresponds to taking the reciprocal of the number raised to the positive of that exponent. So, \( \sqrt{2}^{-4.5} = \frac{1}{\sqrt{2}^{4.5}} \).

3. **Computing \(\sqrt{2}^{4.5}\):** This exponentiates the square root of 2 to the power 4.5, resulting in approximately \( 4.7568 \).

4. **Taking the Reciprocal:** Finally, \( \frac{1}{4.7568} \) gives approximately \( 0.2102 \).


### Stop the Runtime
After processing is complete, stop the runtime to clean up resources.

In [19]:
await sequential_runtime.stop_when_idle()

## [Magentic Chat](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-orchestration/magentic?source=recommendations&pivots=programming-language-python)

### Set Up the Magentic Manager
The Magentic manager coordinates the agents, plans the workflow, tracks progress, and synthesizes the final answer. The standard manager (StandardMagenticManager) uses carefully designed prompts and requires a chat completion model that supports structured output.

In [20]:
from semantic_kernel.agents import StandardMagenticManager
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

chatcompletion_service_id = "chatcompletion_service_id"
manager = StandardMagenticManager(chat_completion_service=AzureChatCompletion(service_id=chatcompletion_service_id))
print(manager.task_ledger_facts_prompt)

Below I will present you a request.

Before we begin addressing the request, please answer the following pre-survey to the best of your ability.
Keep in mind that you are Ken Jennings-level with trivia, and Mensa-level with puzzles, so there should be
a deep well to draw from.

Here is the request:

{{$task}}

Here is the pre-survey:

    1. Please list any specific facts or figures that are GIVEN in the request itself. It is possible that
       there are none.
    2. Please list any facts that may need to be looked up, and WHERE SPECIFICALLY they might be found.
       In some cases, authoritative sources are mentioned in the request itself.
    3. Please list any facts that may need to be derived (e.g., via logical deduction, simulation, or computation)
    4. Please list any facts that are recalled from memory, hunches, well-reasoned guesses, etc.

When answering this survey, keep in mind that "facts" will typically be specific names, dates, statistics, etc.
Your answer should use 

### Set Up the Magentic Orchestration
Combine your agents and manager into a MagenticOrchestration object.

In [21]:
from semantic_kernel.agents import MagenticOrchestration

magentic_orchestration = MagenticOrchestration(
    members=agents,
    manager=manager,
    agent_response_callback=agent_response_callback,
)

### Start the Runtime
Start the runtime to manage agent execution.

In [22]:
from semantic_kernel.agents.runtime import InProcessRuntime

magentic_runtime = InProcessRuntime()
magentic_runtime.start()





==> AGENT **ResearchAgent**---
Here is what I was able to gather so far:

1. **ResNet-50** is known for being relatively energy-efficient for image classification tasks. Specific energy metrics for ResNet-50 during training and inference were not found in precise figures in the latest data sets searched. However, it's often listed as a benchmark model in MLPerf, where efficiency and performance are optimized through advanced architectures【4:2†source】.

2. **BERT-base** has data suggesting that adaptive inference can reduce its energy consumption significantly. An increase in computational workload reduction by 35% during sparse training was noted, though this does not directly give the absolute figures of energy usage【6:1†source】.

3. **GPT-2** typically requires extensive energy due to its large model size and capabilities for tasks like text generation. Recent data indicate that the environmental impact, such as CO2 emissions from the energy consumption of these models, remains a

### Invoke the Orchestration
Invoke the orchestration with your complex task. The manager will plan, delegate, and coordinate the agents to solve the problem.

In [23]:
magentic_orchestration_result = await magentic_orchestration.invoke(
    task=(
        "I am preparing a report on the energy efficiency of different machine learning model architectures. "
        "Compare the estimated training and inference energy consumption of ResNet-50, BERT-base, and GPT-2 "
        "on standard datasets (e.g., ImageNet for ResNet, GLUE for BERT, WebText for GPT-2). "
        "Then, estimate the CO2 emissions associated with each, assuming training on an Azure Standard_NC6s_v3 VM "
        "for 24 hours. Provide tables for clarity, and recommend the most energy-efficient model "
        "per task type (image classification, text classification, and text generation)."
    ),
    runtime=magentic_runtime,
)

### Collect Results
Wait for the orchestration to complete and print the final result.

In [24]:
value = await magentic_orchestration_result.get()
print(f"\nFinal result:\n{value}")


Final result:
Based on the research and computations conducted, here is the summary of our findings on the energy efficiency of different machine learning model architectures: ResNet-50, BERT-base, and GPT-2.

### Energy Consumption and CO2 Emissions Table:
Each model was run for 24 hours on an Azure Standard_NC6s_v3 VM. Here are the estimated energy consumption and CO2 emissions:

| Model       | Energy (kWh) | CO2 Emissions (g) |
|-------------|--------------|-------------------|
| ResNet-50   | 6.72         | 2688              |
| BERT-base   | 10.08        | 4032              |
| GPT-2       | 13.44        | 5376              |

### Task Type Recommendations:
- **Image Classification (ResNet-50):** ResNet-50 is the most energy-efficient model for this task.
- **Text Classification (BERT-base):** While BERT-base consumes more energy than ResNet-50, it is specifically designed for text classification tasks and thus recommended for this type.
- **Text Generation (GPT-2):** Although G

### Stop the Runtime
After processing is complete, stop the runtime to clean up resources.

In [25]:
await magentic_runtime.stop_when_idle()

# Teardown

In [26]:
# delete all files
files_to_delete = await project_client.agents.files.list()
files_to_delete_nr = len(files_to_delete.data)

if files_to_delete_nr>0:
    i=0
    print(f"{files_to_delete_nr} files will now be deleted:")
    for f in files_to_delete.data:
        i += 1
        print(f"- File {i} of {files_to_delete_nr}: {f.filename} (id={f.id}) is being deleted...")
        await project_client.agents.files.delete(f.id)
else:
    print("No files to delete")

No files to delete


## Avoiding ***modifying a collection while iterating over it*** for both threads and agents

The code
```
threads_to_delete = project_client.agents.threads.list()
```
returns an async iterator that **lazily** fetches pages of threads.<br/>
But since we're deleting threads as we iterate, the underlying data source is being mutated during iteration. So when the iterator tries to fetch the next page, it hits a missing resource — hence the **ResourceNotFoundError**.<br/><br/>

This is a classic case of *modifying a collection while iterating over it*, which is risky even in synchronous code — and doubly so in async paged APIs.
### The solution
We need to fully materialize the list of threads before deleting anything. That way, the iterator isn’t affected by the deletions

In [27]:
# delete all threads

threads_to_delete = [t async for t in project_client.agents.threads.list()]
i = 0
for t in threads_to_delete:
    i += 1
    print(f"{i} - Thread <{t.id}> is being deleted...")
    await project_client.agents.threads.delete(thread_id=t.id)

1 - Thread <thread_Jn6oaV64EprOUX2zHvGRuhY2> is being deleted...
2 - Thread <thread_cchGyV33pparnNiYk20WsnPG> is being deleted...
3 - Thread <thread_4LUIZYwejIaViFMu6iE82ryU> is being deleted...
4 - Thread <thread_HJfolWEO0Z1O1zSGaqVa1Sry> is being deleted...


In [28]:
# delete all agents

agents_to_delete = [a async for a in project_client.agents.list_agents(limit=100)]
i=0
for a in agents_to_delete:
    i += 1
    print(f"{i} - Agent <{a.id}> is being deleted...")
    await project_client.agents.delete_agent(agent_id=a.id)

1 - Agent <asst_TjeIuVF5Ja3d7aZvRiFYILYF> is being deleted...


# HIC SUN LEONES